In [123]:
import pandas as pd
from operator import itemgetter
import networkx as nx

In [124]:
data = pd.read_csv('pre_survey.csv')

### Nodes

In [125]:
nodes = data[['ID','Name']] #only need these columns, we fit everything to name for ease (only 40 people)
nodes = nodes.rename(columns={'Name':'Label'})
print("There are", len(nodes), "nodes.")

There are 39 nodes.


In [126]:
nodes.to_csv('nodes.csv', index=False)

### Edges

In [127]:
working = data.iloc[:,4:] #removes this column
working = working.set_index('Name').drop(columns=['NetID','Last modified time']) #set index for stacking, remove other cols
working = working.stack().rename_axis(['Source','Target']).reset_index() #makes into a stack from matrix
edges = working.rename(columns={0:'Weight'}) #rename columns
edges = edges[edges['Target'].isin(nodes['Label'])] #only include targets in the club
edges.head()

,Source,Target,Weight
0,Nikhil Chinchalkar,Nikhil Chinchalkar,I am this person
1,Nikhil Chinchalkar,Jason Wang,I speak with them at least once a week
2,Nikhil Chinchalkar,Rithya Sriram,I speak with them at least once a week
3,Nikhil Chinchalkar,Carina Lau,I speak with them at least once a week
4,Nikhil Chinchalkar,Jenny Williams,I speak with them at least once a week


In [128]:
len(edges['Source'].unique()), len(edges['Target'].unique()) #sanity check, should be 40 people still

(39, 39)

In [129]:
edges['Weight'].unique() #used for mapping weights below

array(['I am this person', 'I speak with them at least once a week',
       "I've spoken to them more than once before",
       'I recognize their face/name', "I've spoken to them once before",
       "I've never seen/heard of this person before",
       'I speak with them everyday'], dtype=object)

In [130]:
weights_map = {'I am this person':0,
               'I speak with them everyday':6,
               'I speak with them at least once a week':5,
               "I've spoken to them more than once before":4,
               "I've spoken to them once before":3,
               'I recognize their face/name':2,
               "I've never seen/heard of this person before":0} #subject to change

In [131]:
edges['Weight'] = edges['Weight'].map(lambda x: weights_map[x])

In [132]:
edges.to_csv('edges.csv', index=False)

### Demographics

In [133]:
demographics = pd.read_csv('demographics.csv')
demographics = pd.merge(demographics, nodes, left_on='Full Name', right_on='Label', how='right')
demographics.to_csv('node_demographics.csv', index=False)

### Graph

In [134]:
demographics.columns

Index(['Role', 'Full Name', 'First Name', 'Last Name', 'Net ID',
       'Cornell email', 'CDJ Join Class', 'Graduation Year', 'Birthday',
       'Major(s)', 'Role Type', 'Project', 'ID', 'Label'],
      dtype='object')

In [145]:
edges = edges[edges['Weight'] != 0]

In [146]:
G = nx.from_pandas_edgelist(
    edges,
    source='Source',
    target='Target',
    edge_attr=['Weight'],
    create_using=nx.DiGraph())

In [147]:
for _, row in demographics.iterrows():
    node_id = row['Full Name']

    G.add_node(node_id)
    for col in demographics.columns:
        if col != 'Full Name':
            G.nodes[node_id][col] = row[col]

In [148]:
nx.write_gexf(G, "cdj_network.gexf")